# **Part 1:**
**Write it:**

P(churn) wrong: 0.136
P(churn) right: 0.012

They differ by more than 10x. When drop_first=True is applied to a single row containing only “Two year” contract, that category is the only one present and gets treated as the first — so it is dropped. Both Contract columns become 0, which the model reads as Month-to-month (the highest-risk contract type), artificially inflating the churn probability from 0.012 to 0.136. No error is raised, making this a silent but serious bug. The serving encoder (prepare_input) fixes this by using get_dummies without drop_first, then reindexing to the exact training columns.

Parity test: Largest difference = 0.0 — parity test passed on 7,043 customers. The app and the notebook give bit-for-bit identical predictions on every customer.

What-if table:

Change	P(churn)	Delta
Baseline	0.935	—
Contract = One year	0.862	−0.073
Contract = Two year	0.765	−0.170
PaymentMethod = Credit card	0.913	−0.022

The two-year contract lowers risk the most (−0.170), dropping churn probability from 93.5% to 76.5%. However this does NOT mean offering a two-year contract would cause the customer to stay — this is an association learned from data, not a causal effect. Customers who sign two-year contracts may already be more committed; simply offering one to a high-risk customer does not guarantee they will accept it or that it will change their underlying intention to leave. A company could find out through a randomised controlled trial — randomly offering contract upgrades to similar customers and measuring actual churn.

# **PART 2:**

**Write it:**

Surprising behavior: When Internet service is set to “No”, all add-on service selectors disappear instantly — but the churn probability also drops significantly even before any other input changes. This is surprising because it reveals how strongly the model ties internet service type to churn risk: removing internet service (which also removes Fiber optic as an option) alone shifts the prediction dramatically.

A user might enter a customer with tenure = 0 and MonthlyCharges = 0 (a brand new customer with no billing yet). The app derives TotalCharges = tenure × monthly = 0, which is technically correct, but the model was trained with very few such customers and may produce unreliable probability estimates near the extremes. The app gives no warning about this edge case.



# **PART 3 :**
**Write it :**


The first deployment did not succeed — I encountered two errors back to back:

Error 1:

ModuleNotFoundError: No module named 'xgboost'
File "/mount/src/churn-risk-advisor/app.py", line 8, in <module>
    import xgboost as xgb

Fix: The xgboost line in requirements.txt was commented out (# xgboost==2.0.3). Since my final model is XGBoost, I uncommented it and pushed the fix.

Error 2:

xgboost.core.XGBoostError
File "app.py", line 29, in load_artifacts
    model.load_model(str(BASE_DIR / "churn_model.json"))

Fix: My load_artifacts() function was calling model.load_model("churn_model.json") after already loading with joblib.load(). These are two different loading methods and cannot be mixed. I removed the model.load_model() line entirely — joblib.load("churn_model.joblib") is the only call needed. After pushing this fix the app deployed successfully.